# 🗣️ Transcribe + Diarize — "who said what" (CPU-only)

Adds **speaker diarization** on top of the Parakeet transcription: it labels each part
of the transcript with a speaker (Speaker 1 / Speaker 2 / …). Runs **locally, no GPU,
no paid API, no Hugging Face token**.

- **Transcription:** NVIDIA Parakeet TDT via `onnx-asr` (same as your transcription notebook).
- **Diarization:** `sherpa-onnx` — a pyannote segmentation model + a speaker-embedding
  model + clustering. Its models download straight from GitHub (no token gate).
- **Merge:** each transcript segment is tagged with whichever speaker was talking during it.

**To use it:** set `INPUT_FILE` (and `NUM_SPEAKERS` if you know it) in the config cell, then Run All.

---
### Prerequisites
- **Python 3.9+**, and **ffmpeg** on PATH (same as before: `brew install ffmpeg` /
  `sudo apt install -y ffmpeg` / `winget install Gyan.FFmpeg`).

## 1. Install

In [ ]:
%pip install -q "onnx-asr[cpu,hub]" sherpa-onnx soundfile numpy
import onnx_asr, sherpa_onnx
print("onnx-asr + sherpa-onnx", sherpa_onnx.__version__, "ready")

## 2. Download the diarization models (first run only)

Two small models from the sherpa-onnx GitHub releases — a segmentation model (~7 MB) and a speaker-embedding model (~40 MB). Cached in `models/` after the first run.

In [ ]:
import urllib.request, tarfile
from pathlib import Path

MODELS = Path("models"); MODELS.mkdir(exist_ok=True)
SEG_MODEL = MODELS / "sherpa-onnx-pyannote-segmentation-3-0" / "model.onnx"
EMB_MODEL = MODELS / "nemo_en_titanet_small.onnx"

SEG_URL = "https://github.com/k2-fsa/sherpa-onnx/releases/download/speaker-segmentation-models/sherpa-onnx-pyannote-segmentation-3-0.tar.bz2"
EMB_URL = "https://github.com/k2-fsa/sherpa-onnx/releases/download/speaker-recongition-models/nemo_en_titanet_small.onnx"

if not SEG_MODEL.exists():
    print("downloading segmentation model ...")
    tgz = MODELS / "seg.tar.bz2"
    urllib.request.urlretrieve(SEG_URL, tgz)
    with tarfile.open(tgz, "r:bz2") as t:
        t.extractall(MODELS)
    tgz.unlink()

if not EMB_MODEL.exists():
    print("downloading speaker-embedding model ...")
    urllib.request.urlretrieve(EMB_URL, EMB_MODEL)

assert SEG_MODEL.exists() and EMB_MODEL.exists()
print("models ready:")
print(" ", SEG_MODEL)
print(" ", EMB_MODEL)

## 3. Configure

👇 Set your file. If you **know** how many speakers are in the audio, set `NUM_SPEAKERS` — it noticeably improves accuracy. Leave it `0` to auto-detect.

In [ ]:
from pathlib import Path
import shutil

INPUT_FILE   = "sample.mp4"   # ← your local video/audio file
NUM_SPEAKERS = 0              # exact count if known (e.g. 2 for an interview); 0 = auto-detect
CLUSTER_THRESHOLD = 0.5       # only used when NUM_SPEAKERS = 0. Higher = fewer speakers.
MODEL_NAME   = "nemo-parakeet-tdt-0.6b-v3"   # transcription model

assert shutil.which("ffmpeg"), "ffmpeg not found on PATH."
assert Path(INPUT_FILE).exists(), f"Input file not found: {INPUT_FILE!r}"
print("input       :", INPUT_FILE)
print("num_speakers:", NUM_SPEAKERS or "auto")

## 4. Extract audio → 16 kHz mono WAV

In [ ]:
import subprocess, soundfile as sf

def to_wav_16k_mono(src, dst="audio_16k.wav"):
    subprocess.run(["ffmpeg","-y","-i",src,"-ar","16000","-ac","1",
                    "-c:a","pcm_s16le",dst,"-loglevel","error"], check=True)
    return dst

WAV = to_wav_16k_mono(INPUT_FILE)
info = sf.info(WAV)
DURATION = info.frames / info.samplerate
print(f"{WAV}: {info.samplerate} Hz, {info.channels}ch, {DURATION/60:.1f} min")

## 5. Transcribe (Parakeet + VAD)

First run downloads the Parakeet + Silero models from Hugging Face; cached afterwards.

In [ ]:
import time, onnx_asr

print("loading Parakeet + Silero VAD ...")
vad = onnx_asr.load_vad("silero")
asr = onnx_asr.load_model(MODEL_NAME).with_vad(vad)

print("transcribing ...")
t0 = time.time()
segments = [s for s in asr.recognize(WAV) if s.text.strip()]   # each: .start .end .text
print(f"done: {len(segments)} segments in {time.time()-t0:.1f}s")

## 6. Diarize (sherpa-onnx)

Figures out *when* each speaker is talking.

In [ ]:
import numpy as np, sherpa_onnx, time

diar_config = sherpa_onnx.OfflineSpeakerDiarizationConfig(
    segmentation=sherpa_onnx.OfflineSpeakerSegmentationModelConfig(
        pyannote=sherpa_onnx.OfflineSpeakerSegmentationPyannoteModelConfig(model=str(SEG_MODEL)),
        num_threads=2,
    ),
    embedding=sherpa_onnx.SpeakerEmbeddingExtractorConfig(model=str(EMB_MODEL), num_threads=2),
    clustering=sherpa_onnx.FastClusteringConfig(
        num_clusters=(NUM_SPEAKERS if NUM_SPEAKERS > 0 else -1),
        threshold=CLUSTER_THRESHOLD,
    ),
    min_duration_on=0.3,
    min_duration_off=0.5,
)
diarizer = sherpa_onnx.OfflineSpeakerDiarization(diar_config)

samples, sr = sf.read(WAV, dtype="float32")
if samples.ndim > 1:
    samples = samples[:, 0]
assert sr == diarizer.sample_rate

print("diarizing ...")
t0 = time.time()
diar_result = diarizer.process(samples)
diar_segs = diar_result.sort_by_start_time()   # each: .start .end .speaker
print(f"done: {diar_result.num_speakers} speaker(s), {len(diar_segs)} segments in {time.time()-t0:.1f}s")

## 7. Merge — label each line with a speaker

Assigns every transcript segment the speaker whose diarized turn overlaps it most, then groups consecutive same-speaker lines into turns.

In [ ]:
def _overlap(a0, a1, b0, b1):
    return max(0.0, min(a1, b1) - max(a0, b0))

def speaker_for(st, en, diar):
    best, best_ov = None, 0.0
    for d in diar:
        ov = _overlap(st, en, d.start, d.end)
        if ov > best_ov:
            best_ov, best = ov, d.speaker
    return best

def label(name):
    return f"Speaker {name + 1}" if name is not None else "Speaker ?"

# label each transcript segment, then merge consecutive same-speaker turns
turns = []
for s in segments:
    spk = speaker_for(s.start, s.end, diar_segs)
    text = s.text.strip()
    if turns and turns[-1]["spk"] == spk:
        turns[-1]["end"] = s.end
        turns[-1]["text"] = (turns[-1]["text"] + " " + text).strip()
    else:
        turns.append({"spk": spk, "start": s.start, "end": s.end, "text": text})

print(f"{diar_result.num_speakers} speaker(s), {len(turns)} turns\n" + "-"*60)
for t in turns:
    print(f"[{t['start']:7.2f}] {label(t['spk'])}: {t['text']}")

## 8. Save labeled transcript (.txt and .srt)

In [ ]:
from pathlib import Path

def fmt_ts(sec):
    ms = int(round(sec*1000)); h,ms = divmod(ms,3_600_000); m,ms = divmod(ms,60_000); s,ms = divmod(ms,1000)
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"

stem = Path(INPUT_FILE).stem

txt_path = f"{stem}.diarized.txt"
with open(txt_path, "w", encoding="utf-8") as f:
    for t in turns:
        f.write(f"{label(t['spk'])}: {t['text']}\n\n")

srt_path = f"{stem}.diarized.srt"
with open(srt_path, "w", encoding="utf-8") as f:
    for i, t in enumerate(turns, 1):
        f.write(f"{i}\n{fmt_ts(t['start'])} --> {fmt_ts(t['end'])}\n{label(t['spk'])}: {t['text']}\n\n")

print("saved:")
print(" •", txt_path)
print(" •", srt_path)

## Notes & tuning

- **Set `NUM_SPEAKERS` when you know it.** For a 2-person interview, `NUM_SPEAKERS = 2`
  is much more reliable than auto-detect. Auto mode uses `CLUSTER_THRESHOLD`: raise it to
  merge over-split speakers, lower it to separate voices it's lumping together.
- **`min_duration_on/off`** control how eagerly short segments are kept or bridged; the
  defaults (0.3 / 0.5 s) are a good start.
- **Speaker numbers are arbitrary and per-file.** "Speaker 1" in one run isn't the same
  person as in another. Diarization says *how many* and *when*, not *who* by name.
- **Overlapping speech** (people talking over each other) is the hard case for any
  clustering-based diarizer; expect some smearing at those moments.
- **Higher accuracy option:** `pyannote.audio` is the SOTA diarizer (better on overlaps),
  but it needs a Hugging Face token + accepting model terms, and pulls in PyTorch — more
  friction, especially in CI. sherpa-onnx is the lighter, token-free choice used here.
- **Embedding model:** `nemo_en_titanet_small` works across languages (voice prints are
  largely language-independent). Larger models (e.g. 3D-Speaker ERes2Net) can improve
  separation at the cost of size/speed.

### Next step
Once the labeling looks good on your audio, we fold this into `scripts/transcribe.py`:
add the two model downloads (cached via the existing Hugging Face cache step or a new
cache), run diarization after transcription, and write the labeled `.txt`/`.srt`. The
`.srt` already carries the speaker prefix, so the frontend needs no change.